# Task 9: Anchor-Free Object Detection (YOLOv8-style) Loss Backpropagation

**Objective:** Build high-precision target regressors by writing classification and CIoU (Complete Intersection over Union) box regression loss computations from scratch.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

def calculate_ciou(bboxes1, bboxes2):
    # bboxes format: (N, 4) -> [x1, y1, x2, y2]
    # Area calculation
    area1 = (bboxes1[:, 2] - bboxes1[:, 0]) * (bboxes1[:, 3] - bboxes1[:, 1])
    area2 = (bboxes2[:, 2] - bboxes2[:, 0]) * (bboxes2[:, 3] - bboxes2[:, 1])
    
    # Intersections
    lt = torch.max(bboxes1[:, :2], bboxes2[:, :2])
    rb = torch.min(bboxes1[:, 2:], bboxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, 0] * wh[:, 1]
    
    # Union
    union = area1 + area2 - inter
    iou = inter / union.clamp(min=1e-6)
    
    # Center distances
    ctr1 = (bboxes1[:, :2] + bboxes1[:, 2:]) / 2.0
    ctr2 = (bboxes2[:, :2] + bboxes2[:, 2:]) / 2.0
    center_dist = torch.sum((ctr1 - ctr2) ** 2, dim=-1)
    
    # Smallest enclosing box dimensions
    enclose_lt = torch.min(bboxes1[:, :2], bboxes2[:, :2])
    enclose_rb = torch.max(bboxes1[:, 2:], bboxes2[:, 2:])
    enclose_wh = (enclose_rb - enclose_lt).clamp(min=0)
    enclose_diag = torch.sum(enclose_wh ** 2, dim=-1).clamp(min=1e-6)
    
    # Aspect ratio metrics
    w1, h1 = bboxes1[:, 2] - bboxes1[:, 0], bboxes1[:, 3] - bboxes1[:, 1]
    w2, h2 = bboxes2[:, 2] - bboxes2[:, 0], bboxes2[:, 3] - bboxes2[:, 1]
    v = (4 / (np.pi ** 2)) * torch.pow(torch.atan(w1 / h1.clamp(min=1e-6)) - torch.atan(w2 / h2.clamp(min=1e-6)), 2)
    
    with torch.no_grad():
        alpha = v / ((1.0 - iou) + v).clamp(min=1e-6)
        
    ciou = iou - (center_dist / enclose_diag + alpha * v)
    return 1.0 - ciou

In [ ]:
# Create mock detections and verify loss calculation
pred_boxes = torch.tensor([[10., 10., 50., 50.], [20., 20., 80., 80.]], requires_grad=True)
target_boxes = torch.tensor([[12., 12., 48., 48.], [20., 20., 80., 80.]])

ciou_loss = calculate_ciou(pred_boxes, target_boxes).mean()
ciou_loss.backward()

print("Computed CIoU Loss:       ", ciou_loss.item())
print("Box Gradients w.r.t Loss:\n", pred_boxes.grad)
assert ciou_loss.item() > 0, "Failed: loss should be non-negative!"
print("Success: Box regression loss and gradients generated smoothly!")